In [ ]:
# Flight Delay Prediction Notebook

This notebook uses the included sample CSV for a quick training run. Replace `sample_flights.csv` with the full `flights.csv` dataset if you want to reproduce the full project on the complete data.
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

BASE_DIR = Path.cwd()
DATASET = BASE_DIR / 'sample_flights.csv'

if not DATASET.exists():
    raise FileNotFoundError('sample_flights.csv not found. Run generate_sample_dataset.py first.')

flights = pd.read_csv(DATASET, low_memory=False)
flights = flights.iloc[:100000]
flights = flights.drop(columns=[
    'YEAR', 'FLIGHT_NUMBER', 'AIRLINE', 'DISTANCE', 'TAIL_NUMBER', 'TAXI_OUT',
    'SCHEDULED_TIME', 'DEPARTURE_TIME', 'WHEELS_OFF', 'ELAPSED_TIME', 'AIR_TIME',
    'WHEELS_ON', 'DAY_OF_WEEK', 'TAXI_IN', 'CANCELLATION_REASON'
], errors='ignore')
flights = flights.fillna(flights.mean(numeric_only=True))

result = [1 if value > 15 else 0 for value in flights['ARRIVAL_DELAY']]
flights['result'] = result
flights = flights.drop(columns=['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'ARRIVAL_TIME', 'ARRIVAL_DELAY'], errors='ignore')

X = flights.drop(columns=['result']).to_numpy()
y = flights['result'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]))
